# Statistical Computing + Welfare (Solutions)

## Setup

### 0.1 Imports and paths

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

In [ ]:
"""
Lesson : Statistical Computing
Dataset: Malawi IHS-5 (2019-2020)
Files: hh_mod_a_filt.dta, HH_MOD_B.dta, HH_MOD_C.dta, ihs5_consumption_aggregate.dta
"""

DATA_DIR = Path('../../../data/0_raw/malawi/IHS 5 DATA sample')

print('Input folder:', DATA_DIR)


**Result Interpretation**
This cell prepares required libraries and resolves the dataset folder path.

### 0.2 Load data

In [ ]:
# Load source datasets with categorical labels enabled for readability.
hh = pd.read_stata(DATA_DIR / 'hh_mod_a_filt.dta', convert_categoricals=True)
roster = pd.read_stata(DATA_DIR / 'HH_MOD_B.dta', convert_categoricals=True)

# For this notebook we only need two education fields.
edu = pd.read_stata(
    DATA_DIR / 'HH_MOD_C.dta',
    convert_categoricals=True,
    columns=['case_id', 'PID', 'hh_c08', 'hh_c09']
)
cons = pd.read_stata(DATA_DIR / 'ihs5_consumption_aggregate.dta', convert_categoricals=True)

print('Loaded shapes:')
print('  hh    ', hh.shape)
print('  roster', roster.shape)
print('  edu   ', edu.shape)
print('  cons  ', cons.shape)


**Result Interpretation**
Check shapes to ensure all source files loaded successfully before analysis.

## Data preparation (shared across all exercises)

### 1) Household size and head profile

In [ ]:
# 1) Household size and head profile
# Household size = number of roster records per household.
hh_size = roster.groupby('case_id').size().rename('hh_size')

# Head is relationship code 1 (or label 'HEAD').
is_head = (
    (pd.to_numeric(roster['hh_b04'], errors='coerce') == 1)
    | (roster['hh_b04'].astype('string').str.strip().str.upper() == 'HEAD')
)

head = roster.loc[is_head, ['case_id', 'PID', 'hh_b05a', 'hh_b03']].copy()
head.columns = ['case_id', 'head_pid', 'head_age', 'head_sex']

# Keep these analysis fields clean.
head['head_age'] = pd.to_numeric(head['head_age'], errors='coerce')
# Keep raw head sex values; map only inside sections where needed.


**Result Interpretation**
This defines household size and extracts one head record per household.

### 2) Education and welfare fields

In [ ]:
# 2) Education + welfare fields
# Keep education as labeled categories in shared prep.
head_edu = edu[['case_id', 'PID', 'hh_c08', 'hh_c09']].copy()
head_edu.columns = ['case_id', 'head_pid', 'head_education_proxy', 'head_qualification_code']

# In this dataset, per-capita real consumption is rexpaggpc.
cons['pcrexpagg'] = cons['rexpaggpc'].copy()
cons_sel = cons[['case_id', 'rexpagg', 'pcrexpagg', 'poor']].copy()

# Keep raw poverty labels/codes; map in the section where needed.


**Result Interpretation**
This step standardizes consumption and poverty fields used in all tests.

### 3) Build final analysis table

In [ ]:
# 3) Assemble analysis frame
# Merge all household-level information into one table.
df = (
    hh
    .merge(hh_size, on='case_id', how='left')
    .merge(head, on='case_id', how='left')
    .merge(head_edu, on=['case_id', 'head_pid'], how='left')
    .merge(cons_sel, on='case_id', how='left')
)

# Keep urban/rural as labeled value from source.
df['urban_rural'] = df['reside']

# Convert core numeric analysis columns once.
for col in ['hh_size', 'head_age', 'rexpagg', 'pcrexpagg']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Households: {len(df):,}')
print('Columns from consumption aggregate (aliased): rexpagg, pcrexpagg, poor')
df[['case_id', 'hh_size', 'head_age', 'head_sex', 'rexpagg', 'pcrexpagg', 'poor']].head()


**Result Interpretation**
The merged `df` is the main table for the rest of the notebook.

## Quick visual diagnostics

In [ ]:
# Quick EDA plots before exercises
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 1) Household size histogram
x1 = df['hh_size'].dropna()
axes[0, 0].hist(x1, bins=range(1, int(x1.max()) + 2), color='#4C78A8', edgecolor='white')
axes[0, 0].set_title('Household size distribution')
axes[0, 0].set_xlabel('hh_size')
axes[0, 0].set_ylabel('Households')

# 2) Household size boxplot
axes[0, 1].boxplot(x1, vert=True)
axes[0, 1].set_title('Household size boxplot')
axes[0, 1].set_ylabel('hh_size')

# 3) Per-capita consumption histogram (raw)
x2 = df['pcrexpagg'].dropna()
axes[1, 0].hist(x2.clip(upper=x2.quantile(0.99)), bins=50, color='#F58518', edgecolor='white')
axes[1, 0].set_title('Per-capita consumption (clipped at p99)')
axes[1, 0].set_xlabel('pcrexpagg')
axes[1, 0].set_ylabel('Households')

# 4) Per-capita consumption by urban/rural (boxplot)
grp_u = df.loc[df['urban_rural'] == 'URBAN', 'pcrexpagg'].dropna()
grp_r = df.loc[df['urban_rural'] == 'RURAL', 'pcrexpagg'].dropna()
axes[1, 1].boxplot([grp_u, grp_r], labels=['URBAN', 'RURAL'], showfliers=False)
axes[1, 1].set_title('Per-capita consumption by urban/rural')
axes[1, 1].set_ylabel('pcrexpagg')

plt.tight_layout()
plt.show()


**Result Interpretation**
These plots provide baseline distributions before formal tests: household size shape, consumption skewness, and urban-rural differences. Use them to sanity-check later statistical results.

## Exercise 1 — Exploring Distribution Objects

### 1.1 The four methods on a normal distribution


In [ ]:
# TODO: fit a normal distribution to pcrexpagg
# 1) compute mean/std
# 2) create fitted = stats.norm(loc=..., scale=...)
# 3) call rvs, pdf, cdf, ppf and print results

x = df['pcrexpagg'].dropna()
mean_x, std_x = x.mean(), x.std()

fitted = stats.norm(loc=mean_x, scale=std_x)

print('10 simulated values:', fitted.rvs(size=10, random_state=42))
print(f'PDF at mean:  {fitted.pdf(mean_x):.6f}')
print(f'CDF at mean:  {fitted.cdf(mean_x):.4f}')
print(f'90th pctile:  {fitted.ppf(0.90):,.0f}')
print('Any simulated negatives?', (fitted.rvs(size=10000, random_state=42) < 0).any())

In [ ]:
# We can also plot the results
x_plot = x.clip(upper=x.quantile(0.99))
xx = np.linspace(x_plot.min(), x_plot.max(), 400)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(
    x_plot,
    bins=50,
    density=True,
    alpha=0.45,
    color='#4C78A8',
    edgecolor='white',
    label='Observed pcrexpagg (clipped at p99)'
)
ax.plot(xx, fitted.pdf(xx), color='#E45756', lw=2.5, label='Fitted normal PDF')
ax.axvline(mean_x, color='black', ls='--', lw=1.2, label=f'Mean = {mean_x:,.0f}')
ax.set_title('Observed vs fitted normal distribution')
ax.set_xlabel('pcrexpagg')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

**Result Interpretation**
`cdf(mean)` is 0.5 for a normal by symmetry. If simulated negatives appear, it highlights a limitation of the normal assumption for strictly non-negative variables like consumption.

### 1.2 Comparing CDF with actual data

In [ ]:
# TODO: compare theoretical vs observed shares below a threshold
# 1) choose a threshold (e.g., median)
# 2) compute fitted.cdf(threshold)
# 3) compute (x <= threshold).mean() and compare

x = df['pcrexpagg']
threshold = x.median()
fitted = stats.norm(loc=x.mean(), scale=x.std())

theoretical_share = fitted.cdf(threshold)
actual_share = (x <= threshold).mean()

print(f'Threshold (median): {threshold:,.0f}')
print(f'Theoretical share below threshold: {theoretical_share:.4f}')
print(f'Actual share below threshold:      {actual_share:.4f}')
print(f'Gap: {actual_share - theoretical_share:+.4f}')


**Result Interpretation**
A visible gap indicates real consumption is not perfectly normal (typically right-skewed).

## Exercise 2 — Confidence Intervals for the Mean


### 2.1 National CI for per-capita consumption

In [ ]:
# TODO: compute the national 95% CI for mean pcrexpagg
# 1) x = df['pcrexpagg'].dropna(), then n, mean_x, se stats.sem(x)
# 2) use stats.t.interval(0.95, df=n-1, loc=mean_x, scale=se)
# 3) print n, mean, CI, and MOE%

x = df['pcrexpagg'].dropna()
n = len(x)
mean_x = x.mean()
se = stats.sem(x)
ci_low, ci_high = stats.t.interval(0.95, df=n-1, loc=mean_x, scale=se)
moe = ci_high - mean_x

print(f'n = {n:,}')
print(f'Mean:  {mean_x:,.0f}')
print(f'SE:    {se:,.0f}')
print(f'95% CI: ({ci_low:,.0f}, {ci_high:,.0f})')
print(f'MOE:   +/-{moe:,.0f} ({moe/mean_x*100:.2f}%)')



In [ ]:
#How do we save the dataframe into excel and csv and stata?
# FINAL_DATA_DIR = Path('../../data/1_cleaned')
# df.to_excel(FINAL_DATA_DIR / 'final_data.xlsx', index= False)
# df.to_csv(FINAL_DATA_DIR / 'final_data.csv', index= False)
# df.to_stata(FINAL_DATA_DIR / 'final_data.dta')

**Result Interpretation**
This interval estimates where the population mean likely lies. MOE% gives an intuitive precision measure.

### 2.2 CIs by region

In [ ]:
# TODO: compute 95% CI by region
# 1) loop over each region
# 2) for each subset, compute n, mean, se, CI, and MOE%
# 3) print a compact comparison table

rows = []
# We define a variable regions that contains unique values of the 'region' column in the dataframe
# sorted in alphabetical.
regions = sorted(df['region'].dropna().astype('string').unique())

for region in regions:
    subset = df.loc[df['region'].astype('string') == region, 'pcrexpagg'].dropna()
    n = len(subset)
    if n < 2:
        continue
    mean_r = subset.mean()
    se = stats.sem(subset)
    lo, hi = stats.t.interval(0.95, df=n-1, loc=mean_r, scale=se)
    moe_pct = (hi - mean_r) / mean_r * 100
    rows.append({'Region': region, 'n': n, 'Mean': mean_r, 
                 'CI_low': lo, 'CI_high': hi, 'MOE%': moe_pct})

region_ci = pd.DataFrame(rows).sort_values('Region')

region_ci

In [ ]:
# We now create a function that make the above code reusable with
# 1. Other dataframe i.e. a generic data
# 2. Other column names i.e variable_name. For instance we want to calculate the CI on 'district' rather than 'region'
# 3. Other numeric columns on which we want to calculate the CI e.g. hh_size in place of 'pcrexpagg'

# Note the `numeric_data_name` argument in the function takes 'pcrexpagg' as default value
def calc_ci(data: pd.DataFrame, 
            variable_name: str, 
            numeric_data_name: str = 'pcrexpagg') -> pd.DataFrame:
    rows = []
    var_values = data[variable_name].unique()
    for var_value in var_values:
        mask = data[variable_name].astype(str) == var_value
        subset = data.loc[mask, numeric_data_name].dropna()

        nobs = len(subset)
        if nobs < 2:
            continue

        mean_r = subset.mean()
        se = stats.sem(subset)
        lo, hi = stats.t.interval(0.95, df=nobs - 1, loc=mean_r, scale=se)
        moe_pct = (hi - mean_r) / mean_r * 100
        row = {
            'variable': var_value,
            'nobs': nobs,
            'mean': mean_r,
            'ci_low': lo,
            'ci_high': hi,
            'moe_pct': moe_pct
        }
        rows.append(row)
        
    return pd.DataFrame(rows)

# We can now call the function on other values, for example: 'district' and 'hh_size'
calc_ci(df, 'district', 'hh_size')

In [ ]:
# We can also call the same function on distinct subsets, for example region and calculate on 'reside' and 'hh_size'
for region in df['region'].unique():
    print(f'Region: {region} \n', calc_ci(df[df['region'] == region], 'reside', 'hh_size'))

**Result Interpretation**
Wider CIs usually reflect fewer observations and/or higher dispersion. Compare overlap carefully when discussing regional gaps.

### 2.3 Urban vs rural CI comparison

In [ ]:
ur = df['urban_rural'].astype('string').str.strip().str.upper().replace({'1': 'URBAN', '2': 'RURAL'})
for g in ['URBAN', 'RURAL']:
    subset = df.loc[ur == g, 'pcrexpagg'].dropna()
    n = len(subset)
    mean_g = subset.mean()
    se = stats.sem(subset)
    lo, hi = stats.t.interval(0.95, df=n-1, loc=mean_g, scale=se)
    print(f"{g}: n={n:,}, mean={mean_g:,.0f}, 95% CI=({lo:,.0f}, {hi:,.0f})")


**Result Interpretation**
If CIs are clearly separated, the group difference is likely statistically meaningful.

## Exercise 3 — Confidence Level and Sample Size


### 3.1 Changing the confidence level

In [ ]:
x = df['pcrexpagg'].dropna()
n = len(x)
mean_x = x.mean()
se = stats.sem(x)

rows = []
for cl in [0.90, 0.95, 0.99]:
    lo, hi = stats.t.interval(cl, df=n-1, loc=mean_x, scale=se)
    rows.append({'Confidence level': f'{int(cl*100)}%', 'CI lower': lo, 'CI upper': hi, 'Width': hi - lo})

pd.DataFrame(rows)


**Result Interpretation**
Higher confidence gives wider intervals: more certainty requires less precision.

### 3.2 Effect of sample size on precision

In [ ]:
x = df['pcrexpagg'].dropna()
results = []
for n in [100, 500, 1000, 5000]:
    s = x.sample(n=n, random_state=42)
    mean_s = s.mean()
    se_s = stats.sem(s)
    lo, hi = stats.t.interval(0.95, df=n-1, loc=mean_s, scale=se_s)
    results.append({'Sample size': n, 'Mean': mean_s, 'MOE%': (hi - mean_s) / mean_s * 100})

# full sample
n_full = len(x)
mean_f = x.mean()
se_f = stats.sem(x)
lo_f, hi_f = stats.t.interval(0.95, df=n_full-1, loc=mean_f, scale=se_f)
results.append({'Sample size': n_full, 'Mean': mean_f, 'MOE%': (hi_f - mean_f) / mean_f * 100})

pd.DataFrame(results)


**Result Interpretation**
MOE shrinks roughly with 1/sqrt(n): large sample gains matter most when n is small.

## Exercise 4 (stretch) — CI for a Proportion


### 4.1 Poverty rate with confidence interval

In [ ]:
poor_txt = df['poor'].astype('string').str.strip().str.lower()
poor_num = poor_txt.map({'poor': 1, 'non-poor': 0, 'non poor': 0})
poor_num = poor_num.fillna(pd.to_numeric(df['poor'], errors='coerce'))

x = poor_num.dropna()
n = len(x)
p_hat = x.mean()
se = np.sqrt(p_hat * (1 - p_hat) / n)
lo, hi = stats.norm.interval(0.95, loc=p_hat, scale=se)

print(f"Estimated poverty rate: {p_hat*100:.2f}% (95% CI: {lo*100:.2f}% - {hi*100:.2f}%)")


**Result Interpretation**
For large samples, normal approximation is standard for proportion CIs. This gives an interpretable uncertainty band around poverty incidence.

### 4.2 Poverty rate by urban/rural + difference CI

In [ ]:
tmp = df[['urban_rural', 'poor']].copy()
tmp['urban_rural'] = tmp['urban_rural'].astype('string').str.strip().str.upper().replace({'1': 'URBAN', '2': 'RURAL'})

poor_txt = tmp['poor'].astype('string').str.strip().str.lower()
tmp['poor_num'] = poor_txt.map({'poor': 1, 'non-poor': 0, 'non poor': 0})
tmp['poor_num'] = tmp['poor_num'].fillna(pd.to_numeric(tmp['poor'], errors='coerce'))

for g in ['URBAN', 'RURAL']:
    s = tmp.loc[tmp['urban_rural'] == g, 'poor_num'].dropna()
    n = len(s)
    p = s.mean()
    se = np.sqrt(p * (1 - p) / n)
    lo, hi = stats.norm.interval(0.95, loc=p, scale=se)
    print(f"{g}: {p*100:.2f}% (95% CI: {lo*100:.2f}% - {hi*100:.2f}%), n={n:,}")

r = tmp.loc[tmp['urban_rural'] == 'RURAL', 'poor_num'].dropna()
u = tmp.loc[tmp['urban_rural'] == 'URBAN', 'poor_num'].dropna()
p_r, p_u = r.mean(), u.mean()
se_diff = np.sqrt(p_r*(1-p_r)/len(r) + p_u*(1-p_u)/len(u))
z = stats.norm.ppf(0.975)
diff = p_r - p_u
lo_d, hi_d = diff - z*se_diff, diff + z*se_diff
print(f"\nDifference (RURAL - URBAN): {diff*100:.2f} pp (95% CI: {lo_d*100:.2f}, {hi_d*100:.2f})")


**Result Interpretation**
If the difference CI excludes zero, poverty rates differ significantly between urban and rural groups.

## Exercise 5 — Outlier Detection (5.5)

### 5.1 IQR fences on household size

In [ ]:
x = df['hh_size'].dropna()
q1, q3 = x.quantile(0.25), x.quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

iqr_flag_hhsize = (x < lower) | (x > upper)
print(f'Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}')
print(f'Lower={lower:.2f}, Upper={upper:.2f}')
print(f'Flagged: {iqr_flag_hhsize.sum()} ({iqr_flag_hhsize.mean():.1%})')
print(f'Mean (all): {x.mean():.2f}  -> Mean (clean): {x[~iqr_flag_hhsize].mean():.2f}')
print(f'Median (all): {x.median():.1f}  -> Median (clean): {x[~iqr_flag_hhsize].median():.1f}')


**Result Interpretation**
IQR fences identify unusually large/small household sizes. Compare all-vs-clean mean/median to see whether extreme values meaningfully influence summary statistics.

### 5.1 Plot task

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].boxplot(x)
ax[0].set_title('hh_size: all data')
ax[1].boxplot(x[~iqr_flag_hhsize])
ax[1].set_title('hh_size: after IQR filter')
for a in ax:
    a.set_ylabel('hh_size')
plt.tight_layout(); plt.show()

**Result Interpretation**
These boxplots visualize how strongly IQR filtering changes the household-size distribution. Large change in whiskers/outliers with stable median suggests robust central tendency.

### 5.2 MAD-based robust z-scores on per-capita consumption

In [ ]:
tmp = df[['case_id', 'pcrexpagg', 'region', 'urban_rural']].dropna().copy()
med = tmp['pcrexpagg'].median()
mad = stats.median_abs_deviation(tmp['pcrexpagg'], scale='normal')
tmp['z_score'] = (tmp['pcrexpagg'] - med) / mad
mad_flag_pc = tmp['z_score'].abs() > 3.5

print(f'Median: {med:.2f}')
print(f'MAD(normal): {mad:.2f}')
print(f'Flagged: {mad_flag_pc.sum()} ({mad_flag_pc.mean():.1%})')

extreme = tmp.loc[mad_flag_pc, ['case_id', 'pcrexpagg', 'z_score', 'region', 'urban_rural']].sort_values('pcrexpagg', ascending=False)
extreme.head(10)


**Result Interpretation**
MAD-based robust z-scores flag extreme per-capita consumption values less sensitively to skew than standard z-scores. Review the top extremes to assess plausibility and potential data issues.

### 5.2 Plot task

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(tmp['z_score'].clip(-20, 20), bins=80, color='#F58518', edgecolor='white')
ax.axvline(-3.5, color='red', linestyle='--', linewidth=1)
ax.axvline(3.5, color='red', linestyle='--', linewidth=1)
ax.set_title('Robust z-scores for pcrexpagg (clipped to [-20,20])')
ax.set_xlabel('robust z')
ax.set_ylabel('Households')
plt.tight_layout(); plt.show()


**Result Interpretation**
This histogram shows how far robust z-scores extend into tails and where the ±3.5 cutoffs sit. Heavy right-tail mass indicates high-end welfare outliers are driving dispersion.

In [ ]:
flag_ur = (tmp.loc[mad_flag_pc, 'urban_rural']
           .astype('string').str.strip().str.upper()
           .value_counts(normalize=True).mul(100).round(2))
print('Share of MAD-flagged outliers by urban/rural (%):')
print(flag_ur)


**Result Interpretation**
This table shows whether flagged consumption outliers are concentrated in urban or rural households. Concentration patterns can indicate structural differences, not only data errors.

### 5.3 Before-and-after comparison table for pcrexpagg

In [ ]:
x = df['pcrexpagg'].dropna()

# IQR filter for pcrexpagg
q1, q3 = x.quantile(0.25), x.quantile(0.75)
iqr = q3 - q1
iqr_flag_pc = (x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)

# MAD filter for pcrexpagg
med = x.median()
mad = stats.median_abs_deviation(x, scale='normal')
z = (x - med) / mad
mad_flag_pc2 = z.abs() > 3.5

summary = pd.DataFrame({
    'Metric': ['n', 'Mean', 'Median', 'Std'],
    'All data': [x.size, x.mean(), x.median(), x.std()],
    'After IQR filter': [x[~iqr_flag_pc].size, x[~iqr_flag_pc].mean(), x[~iqr_flag_pc].median(), x[~iqr_flag_pc].std()],
    'After MAD filter': [x[~mad_flag_pc2].size, x[~mad_flag_pc2].mean(), x[~mad_flag_pc2].median(), x[~mad_flag_pc2].std()],
})
summary


**Result Interpretation**
This table quantifies sensitivity of headline welfare metrics to outlier filtering. If mean shifts more than median, outliers mainly affect tail-sensitive statistics.

### 5.3 Plot task

In [ ]:
mean_row = summary.loc[summary['Metric'] == 'Mean', ['All data', 'After IQR filter', 'After MAD filter']].iloc[0]
fig, ax = plt.subplots(figsize=(7, 4))
mean_row.plot(kind='bar', ax=ax, color=['#4C78A8', '#72B7B2', '#54A24B'])
ax.set_title('Mean pcrexpagg before/after outlier filtering')
ax.set_ylabel('Mean pcrexpagg')
plt.tight_layout(); plt.show()


**Result Interpretation**
The bar chart makes mean shifts visually comparable across filtering choices. Use it to justify whether reporting median (or robust mean) may be more stable for policy communication.

## Exercise 6 — Hypothesis Testing (5.6)

### 6.1 Urban vs rural per-capita consumption (Welch t-test)

In [ ]:
ur = df['urban_rural'].astype('string').str.strip().str.upper()
urban = df.loc[ur == 'URBAN', 'pcrexpagg'].dropna()
rural = df.loc[ur == 'RURAL', 'pcrexpagg'].dropna()
print(f'Urban: n={len(urban)}, mean={urban.mean():.2f}, std={urban.std():.2f}')
print(f'Rural: n={len(rural)}, mean={rural.mean():.2f}, std={rural.std():.2f}')

t_stat, p_val = stats.ttest_ind(urban, rural, equal_var=False)
diff = urban.mean() - rural.mean()
print(f'Welch t-test: t={t_stat:.4f}, p={p_val:.6g}')
print(f'Difference (urban-rural): {diff:.2f} ({diff / rural.mean() * 100:.2f}% of rural mean)')


**Result Interpretation**
Welch t-test compares average per-capita consumption between urban and rural groups allowing unequal variances. Interpret both significance (p-value) and effect size (difference in Kwacha/%).

### 6.1 Plot task

In [ ]:
means = pd.Series({'URBAN': urban.mean(), 'RURAL': rural.mean()})
ses = pd.Series({'URBAN': urban.std() / np.sqrt(len(urban)), 'RURAL': rural.std() / np.sqrt(len(rural))})
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(means.index, means.values, yerr=1.96 * ses.values, capsize=5, color=['#E45756', '#4C78A8'])
ax.set_title('Mean pcrexpagg by urban/rural (95% CI)')
ax.set_ylabel('pcrexpagg')
plt.tight_layout(); plt.show()


**Result Interpretation**
This mean-with-CI chart complements the t-test by showing uncertainty and overlap directly. It helps communicate practical magnitude beyond statistical significance.

### 6.2 Non-parametric check (Mann-Whitney)

In [ ]:
u_stat, p_u = stats.mannwhitneyu(urban, rural, alternative='two-sided')
print(f'Mann-Whitney U: U={u_stat:.0f}, p={p_u:.6g}')


**Result Interpretation**
Mann-Whitney provides a distribution-robust check for group differences. Agreement with Welch strengthens confidence; disagreement signals non-normality or outlier sensitivity.

### 6.3 Urban vs rural household size (second t-test)

In [ ]:
ur = df['urban_rural'].astype('string').str.strip().str.upper()
u_hh = df.loc[ur == 'URBAN', 'hh_size'].dropna()
r_hh = df.loc[ur == 'RURAL', 'hh_size'].dropna()

th, ph = stats.ttest_ind(u_hh, r_hh, equal_var=False)
dh = u_hh.mean() - r_hh.mean()
print(f'Urban hh_size mean={u_hh.mean():.3f}, Rural hh_size mean={r_hh.mean():.3f}')
print(f'Welch t-test hh_size: t={th:.4f}, p={ph:.6g}')
print(f'Difference (urban-rural): {dh:.3f}')


**Result Interpretation**
This second t-test checks whether household-size differences align with welfare differences. Directional consistency (smaller urban households, higher per-capita welfare) supports the narrative.

### 6.4 Chi-square: poverty status × urban/rural

In [ ]:
sub = df[['urban_rural', 'poor']].copy()
sub['urban_rural'] = sub['urban_rural'].astype('string').str.strip().str.upper()
sub = sub[sub['urban_rural'].isin(['URBAN', 'RURAL'])].copy()

# Local mapping for poverty (labels or numeric codes)
poor_txt = sub['poor'].astype('string').str.strip().str.lower()
sub['poor_num'] = poor_txt.map({'poor': 1, 'non-poor': 0, 'non poor': 0})
sub['poor_num'] = sub['poor_num'].fillna(pd.to_numeric(sub['poor'], errors='coerce'))
sub = sub.dropna(subset=['poor_num']).copy()
sub['poor_num'] = sub['poor_num'].astype(int)

table = pd.crosstab(sub['urban_rural'], sub['poor_num'])
chi2, p, dof, expected = stats.chi2_contingency(table)

print(table)
print(f'\nChi-square: {chi2:.2f}, p-value: {p:.6g}, dof: {dof}')
print('\nRow percentages:')
print(pd.crosstab(sub['urban_rural'], sub['poor_num'], normalize='index').round(3))


**Result Interpretation**
Chi-square tests association between poverty status and urban/rural residence. Row percentages are key for interpretation: they show how poverty burden differs by place type.

### 6.4 Plot task

In [ ]:
row_pct = pd.crosstab(sub['urban_rural'], sub['poor_num'], normalize='index')
fig, ax = plt.subplots(figsize=(7, 4))
row_pct.plot(kind='bar', stacked=True, ax=ax, color=['#72B7B2', '#E45756'])
ax.set_title('Poverty composition by urban/rural (row %)')
ax.set_ylabel('Share')
ax.legend(title='poor_num')
plt.tight_layout(); plt.show()


**Result Interpretation**
The stacked composition chart visualizes poverty-rate contrast between urban and rural households. It is useful for communicating chi-square findings to non-technical audiences.

### 6.5 One-sample t-test benchmark (mean hh_size = 4.4)

In [ ]:
x = df['hh_size'].dropna()
t1, p1 = stats.ttest_1samp(x, popmean=4.4)
print(f'One-sample t-test: t={t1:.4f}, p={p1:.6g}')
print(f'Sample mean={x.mean():.4f} vs benchmark=4.4')


**Result Interpretation**
This one-sample t-test checks whether observed mean household size differs from the benchmark 4.4. A small, non-significant difference supports benchmark consistency.

## Exercise 7 — Correlation & Simple Regression (5.8)

### 7.1 Household size vs per-capita consumption

In [ ]:
tmp = df[['hh_size', 'pcrexpagg']].dropna()
rp, pp = stats.pearsonr(tmp['hh_size'], tmp['pcrexpagg'])
rs, ps = stats.spearmanr(tmp['hh_size'], tmp['pcrexpagg'])
print(f'Pearson: r={rp:.4f}, p={pp:.6g}')
print(f'Spearman: rho={rs:.4f}, p={ps:.6g}')


**Result Interpretation**
Pearson and Spearman together reveal linear vs monotonic association between household size and welfare. Stronger Spearman magnitude often indicates nonlinear/rank-driven relationships.

### 7.2 Age of head vs per-capita consumption

In [ ]:
tmp = df[['head_age', 'pcrexpagg']].dropna()
rp, pp = stats.pearsonr(tmp['head_age'], tmp['pcrexpagg'])
rs, ps = stats.spearmanr(tmp['head_age'], tmp['pcrexpagg'])
print(f'Pearson: r={rp:.4f}, p={pp:.6g}')
print(f'Spearman: rho={rs:.4f}, p={ps:.6g}')


**Result Interpretation**
These correlations assess whether head age is related to welfare. Compare sign and magnitude with prior expectations and note that statistical significance may coexist with weak effect size.

### 7.3 Simple regression: pcrexpagg ~ hh_size

In [ ]:
tmp = df[['hh_size', 'pcrexpagg']].dropna()
ols = stats.linregress(x=tmp['hh_size'], y=tmp['pcrexpagg'])
print(f'Slope: {ols.slope:.4f}')
print(f'Intercept: {ols.intercept:.2f}')
print(f'R-squared: {ols.rvalue**2:.4f}')
print(f'P-value: {ols.pvalue:.6g}')


**Result Interpretation**
OLS regression quantifies average welfare change per additional household member. Use slope + R² together: statistically significant slopes can still explain limited variance.

### 7.3 Plot task

In [ ]:
tmp = df[['hh_size', 'pcrexpagg']].dropna()
# Use a sample for visibility
plot_df = tmp.sample(min(4000, len(tmp)), random_state=42)
x_line = np.linspace(plot_df['hh_size'].min(), plot_df['hh_size'].max(), 100)
y_line = ols.intercept + ols.slope * x_line

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(plot_df['hh_size'], plot_df['pcrexpagg'], s=8, alpha=0.25, color='#4C78A8')
ax.plot(x_line, y_line, color='red', linewidth=2, label='OLS fit')
ax.set_title('pcrexpagg vs hh_size with OLS fit')
ax.set_xlabel('hh_size')
ax.set_ylabel('pcrexpagg')
ax.legend()
plt.tight_layout(); plt.show()


**Result Interpretation**
This scatter + fitted line shows whether the linear model is visually reasonable and how dispersion changes across household size. Wide spread around line indicates low predictive power.

### 7.4 Robust regression comparison (Theil-Sen)

In [ ]:
tmp = df[['hh_size', 'pcrexpagg']].dropna()
ts_slope, ts_intercept, ts_lo, ts_hi = stats.theilslopes(y=tmp['pcrexpagg'], x=tmp['hh_size'])
print(f'OLS slope: {ols.slope:.4f}')
print(f'Theil slope: {ts_slope:.4f}')
print(f'Theil 95% CI: [{ts_lo:.4f}, {ts_hi:.4f}]')


**Result Interpretation**
Theil-Sen slope offers a robust alternative less affected by extreme values. Divergence from OLS suggests outliers/tails are influencing the classical regression estimate.

In [ ]:
x_line = np.linspace(plot_df['hh_size'].min(), plot_df['hh_size'].max(), 100)
y_ols = ols.intercept + ols.slope * x_line
y_theil = ts_intercept + ts_slope * x_line

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(plot_df['hh_size'], plot_df['pcrexpagg'], s=8, alpha=0.20, color='#999999')
ax.plot(x_line, y_ols, color='red', linewidth=2, label='OLS')
ax.plot(x_line, y_theil, color='green', linewidth=2, label='Theil-Sen')
ax.set_title('OLS vs Theil-Sen regression lines')
ax.set_xlabel('hh_size')
ax.set_ylabel('pcrexpagg')
ax.legend()
plt.tight_layout(); plt.show()


**Result Interpretation**
Overlaying OLS and Theil-Sen lines makes robustness differences explicit. If lines separate meaningfully, prefer robust slope for interpretation under heavy-tail/outlier conditions.

### 7.5 Stretch: correlation matrix

In [ ]:
tmp_df = df.copy()

# Convert education categories to numeric codes locally for correlation only
if pd.api.types.is_categorical_dtype(tmp_df['head_education_proxy']) or tmp_df['head_education_proxy'].dtype == 'object':
    tmp_df['head_education_proxy_num'] = tmp_df['head_education_proxy'].astype('category').cat.codes.replace(-1, np.nan)
else:
    tmp_df['head_education_proxy_num'] = pd.to_numeric(tmp_df['head_education_proxy'], errors='coerce')

cols = ['hh_size', 'head_age', 'pcrexpagg', 'head_education_proxy_num']
corr = tmp_df[cols].corr()
print(corr)

high = []
for i, c1 in enumerate(cols):
    for c2 in cols[i+1:]:
        v = corr.loc[c1, c2]
        if pd.notna(v) and abs(v) > 0.5:
            high.append((c1, c2, float(v)))

print('\n|r| > 0.5 pairs:')
print(high if high else 'None')


**Result Interpretation**
The correlation matrix summarizes pairwise associations among key numeric variables. Focus on sign, magnitude, and whether any pair exceeds your practical threshold.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(cols)))
ax.set_yticks(range(len(cols)))
pretty_cols = ['hh_size', 'head_age', 'pcrexpagg', 'head_education_proxy_num']
ax.set_xticklabels(pretty_cols, rotation=45, ha='right')
ax.set_yticklabels(pretty_cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha='center', va='center', color='black', fontsize=9)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Correlation matrix heatmap')
plt.tight_layout(); plt.show()


**Result Interpretation**
The heatmap improves pattern recognition in the matrix and helps compare strengths at a glance. Use it to support the narrative in the written discussion.